<small>Copyright (c) 2026 Advanced Micro Devices, Inc. Licensed under the Apache License, Version 2.0; see the `LICENSE` file in the repository root.</small>

# Tutorial 2 — Decoding a qLDPC code on multiple GPUs

This notebook extends the [single-GPU quickstart](01_single_gpu_quickstart.ipynb)
to multiple GPUs:

1. Build a decoding problem for the [[72,12,6]] bivariate bicycle code.
2. Decode a batch using one worker process per GPU.
3. Check the syndrome solutions and logical error rate.
4. Compare throughput with 1, 2, 4, and 8 GPUs, keeping 1,024 shots per GPU.

Pelagos's process backend keeps one worker process per GPU and splits
each batch across them. Each worker runs the existing Triton decoder. Reusing the
decoder keeps those workers alive between calls.

**Requirements:** at least two AMD GPUs with ROCm on the same machine, and a clone of
this repository with the tutorial dependencies. Use a ROCm PyTorch/Triton environment,
then run from the clone:

```bash
pip install -e ".[tutorials]"
jupyter lab tutorials/
```

Run cells in order. First decodes include compilation and autotuning and may take
longer.


## 0. Setup

GPU IDs below refer to the devices visible to this notebook. Set any GPU visibility
variables before starting Jupyter. The default selects up to eight visible GPUs.
With eight GPUs, the scaling section runs all four counts: 1, 2, 4, and 8.
With fewer GPUs, it runs the available counts.


In [ ]:
import gzip
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import stim
import torch

from pelagos.process import ProcessRelayBPDecoder
from pelagos.stim.sinter.check_matrices import CheckMatrices

assert torch.cuda.is_available(), "An AMD GPU with ROCm is required."
DEVICES = list(range(min(8, torch.cuda.device_count())))
assert len(DEVICES) >= 2 and len(set(DEVICES)) == len(DEVICES)
assert all(0 <= d < torch.cuda.device_count() for d in DEVICES), "Allocate at least two GPUs."

def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "tests/testdata/bicycle_bivariate").is_dir():
            return p
    raise FileNotFoundError("Run this notebook from inside the pelagos clone.")

REPO = find_repo_root()


## 1. Load a circuit and sample shots

As in Tutorial 1, we load a bundled stim circuit at physical error rate 0.003,
then construct its check matrix $H$, observable matrix $O$, and error priors.
Sampling also gives the true logical observable flips for each shot.


In [ ]:
paths = list((REPO / "tests/testdata/bicycle_bivariate").glob(
    "*bicycle_bivariate_72_12_6_memory_X,*error_rate=0.003,*.stim.gz"
))
assert len(paths) == 1, f"Expected one circuit, found {len(paths)}."
circuit = stim.Circuit(gzip.decompress(paths[0].read_bytes()).decode())
dem = circuit.detector_error_model(decompose_errors=False)
cm = CheckMatrices.from_dem(dem)
H = cm.check_matrix.tocsr()
O = cm.observables_matrix.tocsr()
print("Circuit:", paths[0].name)
print("H:", H.shape, "| nonzeros:", H.nnz, "| logical observables:", O.shape[0])

PER_GPU_BATCH = 1024
NUM_SHOTS = PER_GPU_BATCH * len(DEVICES)
detectors, observables, _ = dem.compile_sampler(seed=2026).sample(NUM_SHOTS)
syndromes = np.ascontiguousarray(detectors, dtype=np.uint8)

CONFIG = dict(
    H_csr=H, error_priors=cm.error_priors,
    algo="relay", perf="throughput", data_type="float32",
    pre_iter=80, num_sets=100, set_max_iter=60,
    gamma0=0.65, gamma_dist_interval=(-0.24, 0.66), stop_nconv=1,
    seed=1234, gamma_draw="broadcast", scheduling="flood-fill",
)
print("Decoder settings:", {k: v for k, v in CONFIG.items() if k not in ("H_csr", "error_priors")})


## 2. Decode on multiple GPUs

Pass the syndromes as a NumPy array or CPU tensor of shape `[batch, checks]`.
The workers decode their assigned shots on the GPUs and return the results in
the original input order.

With 1,024 shots per GPU and eight GPUs, the global batch contains 8,192 shots.
Reuse the decoder across batches. The `with` block automatically closes its
worker processes when the block finishes.


In [ ]:
with ProcessRelayBPDecoder(**CONFIG, device=DEVICES) as decoder:
    decoder.decode(syndromes)  # Warm this batch shape; workers persist.
    result = decoder.decode(syndromes)

# Returned CPU tensors remain valid after the decoder is closed.
assert result["errors"].device.type == "cpu"
errors = result["errors"].numpy()
assert errors.shape == (NUM_SHOTS, H.shape[1])
assert np.all((errors == 0) | (errors == 1))
print("Decoded error patterns:", errors.shape)
print("Result fields:", sorted(result))


The result fields have the same meaning as in Tutorial 1:

| key | shape | meaning |
|---|---|---|
| `errors` | `[B, V]` | estimated error pattern for each shot |
| `found_count` | `[B]` | number of valid solutions found; `> 0` means the shot converged |
| `iterations` | `[B]` | BP iterations spent on each shot |
| `weights` | `[B]` | negative log-likelihood of the returned solution |

The GPU workers return the decoded results in the original input order.

## 3. Check the result

A syndrome is satisfied when $H\hat e=s\pmod 2$. Logical decoding succeeds when
$O\hat e$ matches the sampled observable flips. These are different checks: a valid
syndrome solution can still have a logical error. We report both, plus the fraction
of shots for which the decoder found a valid solution.


In [ ]:
satisfied = np.all(((H @ errors.T) % 2).T == syndromes, axis=1)
predicted_observables = ((O @ errors.T) % 2).T.astype(bool)
logical_failures = np.any(predicted_observables != observables, axis=1)
converged = result["found_count"].numpy() > 0
failure_count = int(logical_failures.sum())
print(f"Syndrome satisfied: {satisfied.mean():.3%}")
print(f"Converged:          {converged.mean():.3%}")
print(f"Logical failures:   {failure_count}/{NUM_SHOTS}")
print(f"Logical error rate: {failure_count / NUM_SHOTS:.5f} per shot (all 12 logical qubits, 6 rounds)")


Decoding succeeds when the predicted observable flips match the true flips.

## 4. Scale from one to eight GPUs

We keep 1,024 shots per GPU and increase the global batch with
GPU count. For 1, 2, 4, and 8 GPUs, the global batches are 1,024, 2,048, 4,096,
and 8,192 shots. Every run uses the process backend, including the one-GPU baseline.

Each setting gets a warm-up at its actual batch shape, followed by three measured
repetitions. Workers are reused. The timer covers the blocking
`decode()` calls: host dispatch, transfers, GPU decoding, and returned CPU results.
Sampling, worker startup, warm-up, and quality calculations are outside the timer.


In [ ]:
GPU_COUNTS = [n for n in (1, 2, 4, 8) if n <= len(DEVICES)]
REPEATS = 3
# Reuse prefixes of the sampled shots as the global batch grows.
rows = []

for count in GPU_COUNTS:
    batch_size = PER_GPU_BATCH * count
    batch = syndromes[:batch_size]
    with ProcessRelayBPDecoder(**CONFIG, device=DEVICES[:count]) as decoder:
        decoder.decode(batch)  # Warm up once at this batch shape.
        seconds = []
        for _ in range(REPEATS):
            started = time.perf_counter()
            output = decoder.decode(batch)
            seconds.append(time.perf_counter() - started)
    rates = batch_size / np.asarray(seconds)
    rows.append(dict(gpus=count, batch=batch_size,
                     median=float(np.median(rates)), low=float(rates.min()),
                     high=float(rates.max()), seconds=seconds))

baseline = rows[0]["median"]
for row in rows:
    row["speedup"] = row["median"] / baseline

print(f"{'GPUs':>4} {'shots/GPU':>10} {'global batch':>12} {'shots/s':>10} {'speedup':>8}")
for row in rows:
    print(f"{row['gpus']:>4} {PER_GPU_BATCH:>10} {row['batch']:>12} "
          f"{row['median']:>10.0f} {row['speedup']:>7.2f}x")
    print("  measured seconds:", [round(t, 4) for t in row["seconds"]])


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
y = np.array([r["speedup"] for r in rows])
low = np.array([r["low"] / baseline for r in rows])
high = np.array([r["high"] / baseline for r in rows])
ax.errorbar(GPU_COUNTS, y, yerr=[y - low, high - y],
            marker="o", capsize=3, label="multi-GPU scaling")
ax.plot(GPU_COUNTS, GPU_COUNTS, "k--", alpha=0.4, label="ideal linear scaling")
ax.set(xticks=GPU_COUNTS, xlabel="GPUs", ylabel="Throughput / one-GPU median",
       title="Scaling: 1,024 shots per GPU")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


The points show median throughput relative to the corresponding one-GPU median.
Bars show the observed minimum and maximum rates divided by that same baseline.

Each additional GPU handles another 1,024 shots. The speedup compares the total
shots decoded per second with the one-GPU run. The dashed line shows proportional
growth: twice as many GPUs would give twice the aggregate throughput.

For the full multi-GPU design and performance evaluation, see our paper,
*Scaling Pêlagos: Multi-GPU Orchestration for Relay Belief-Propagation Decoding*.

## Next steps

* To select particular GPUs, edit `DEVICES`, for example `[0, 2, 4, 6]`.
  `GPU_COUNTS` selects how many of those devices each run uses.
* Lower `PER_GPU_BATCH` if GPU memory is limited. Larger codes also need smaller
  per-GPU batches; keep `H.nnz × per-GPU batch` below the decoder's indexing limit.
* For the higher-level `RelayDecoder` / `ObservableDecoderRunner` API, see
  [process-backend documentation](../docs/process-backend.md).
